# A check can read only declared context fields

The check declares user.user_id, then accidentally tries to read request.client_ip.

Run with the Python environment of this checkout (`.venv`). These examples use the API in this branch; installing a released package may give a different API.

[Script](10_context_in_the_check.py) · [Tutorial](../../docs/tutorials/step-03-authorization-and-roles.md)

## Imports

In [1]:
"""A check can read only declared context fields."""

from __future__ import annotations

import asyncio

from pydantic import Field

from aoa.action_machine.auth import ApplicationRole
from aoa.action_machine.context import Context
from aoa.action_machine.context.context_view import ContextView
from aoa.action_machine.context.user_info import UserInfo
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.access_control import Allowed, Verdict
from aoa.action_machine.intents.access_decide import access_decide
from aoa.action_machine.intents.aspects import summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.context_requires import context_requires
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.action_machine.resources import BaseResource
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine
from aoa.action_machine.runtime.tools_box import ToolsBox

## The rule

The check declares user.user_id, then accidentally tries to read request.client_ip.

In [2]:
from aoa.action_machine.model.base_state import BaseState


class StoreDomain(BaseDomain):
    """Group the order operations."""

    name = "store"
    description = "Order management"


class ManagerRole(ApplicationRole):
    """Permit order management."""

    name = "manager"
    description = "Can manage orders"


class OrderParams(BaseParams):
    """Identify the order to cancel."""

    order_id: str = Field(description="Order identifier")


class OrderResult(BaseResult):
    """Report the cancelled order."""

    order_id: str = Field(description="Cancelled order identifier")


@meta(description="Cancel an order", domain=StoreDomain)
@check_roles(ManagerRole)
class CancelOrderAction(BaseAction[OrderParams, OrderResult]):
    """Cancel an order after checking access."""

    @access_decide("Check whether the order can be cancelled")
    @context_requires("user.user_id")
    async def cancel_access_decide(
        self,
        params: OrderParams,
        box: ToolsBox,
        connections: Connections,
        ctx: ContextView,
    ) -> Verdict:
        """Decide access before cancelling the order."""
        print("declared user:", ctx.get("user.user_id"))
        ctx.get("request.client_ip")
        return Allowed()

    @summary_aspect("Cancel the order")
    async def cancel_summary(self, params: OrderParams, state: BaseState, box: ToolsBox, connections: Connections) -> OrderResult:
        """Return the cancellation result."""
        return OrderResult(order_id=params.order_id)

## Try it

Run the cells in order. The calls below are the same as in the script.

In [3]:
machine = ActionProductMachine(cache_coordinator=None)
caller = Context(user=UserInfo(user_id="m-1", roles=(ManagerRole,)))

In [4]:
answer = await machine.check_access_decide(caller, CancelOrderAction, OrderParams(order_id="ord-001"))
print("answer:", answer.model_dump(mode="json"))
print("server cause:", type(answer.cause).__name__)

declared user: m-1
answer: {'kind': 'undecided', 'gate': 'ACCESS_DECIDE'}
server cause: ContextAccessError


## What the result means

The declared user ID is readable. The undeclared key raises ContextAccessError, so access remains undecided. Add a key to context_requires only when the check actually needs it.